## Task 13: Direct Preference Optimization (DPO)
The real spec needs PyTorch + TRL with an actively-trained policy model and a frozen reference model. That's not runnable here, but the DPO loss itself is just a formula over log-probabilities — the cell below computes it directly on toy log-prob values so you can see exactly what the loss is doing, followed by the full TRL-based reference training script.

In [ ]:
!pip install trl -q

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U transformers datasets trl accelerate

In [ ]:
import torch
import transformers
import datasets
import trl

print("GPU:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)

In [ ]:
!pip show trl
!pip list | grep trl

In [ ]:
import trl
print("TRL version:", trl.__version__)

In [ ]:
!pip install -U trl transformers accelerate datasets -q

In [ ]:
import numpy as np

def dpo_loss(logp_chosen_policy, logp_rejected_policy,
             logp_chosen_ref, logp_rejected_ref, beta=0.1):
    policy_logratio = logp_chosen_policy - logp_rejected_policy
    ref_logratio = logp_chosen_ref - logp_rejected_ref
    logits = beta * (policy_logratio - ref_logratio)
    loss = -np.log(1 / (1 + np.exp(-logits)) + 1e-9)
    return loss, logits

logp_chosen_policy  = np.array([-1.2])
logp_rejected_policy = np.array([-2.5])
logp_chosen_ref     = np.array([-1.5])
logp_rejected_ref   = np.array([-1.7])

loss, logits = dpo_loss(logp_chosen_policy, logp_rejected_policy,
                         logp_chosen_ref, logp_rejected_ref, beta=0.1)
print("DPO logits (implicit reward margin):", logits)
print("DPO loss:", loss)

In [ ]:
!pip show trl

In [ ]:
!pip uninstall trl -y -q
!pip cache purge

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU not available")

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

In [ ]:
# Uninstall all potentially conflicting packages to ensure a clean slate
!pip uninstall numpy trl transformers accelerate datasets scikit-learn scipy -y -q

# Install scikit-learn first to let it pull in compatible numpy/scipy versions
!pip install scikit-learn -q

# Then install the rest of the libraries
!pip install trl transformers accelerate datasets -q

import trl
print("TRL version after re-installation:", trl.__version__)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

model_name = "gpt2"
policy_model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

pref_data = Dataset.from_dict({
    "prompt": ["Explain gravity."],
    "chosen": ["Gravity is the force that attracts two masses toward each other."],
    "rejected": ["Gravity is when things fall because they feel like it."],
})

config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    num_train_epochs=3,
    max_length=128,
    report_to=[],
    use_cpu=True,
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=config,
    train_dataset=pref_data,
    processing_class=tokenizer,   # renamed from tokenizer= in current TRL
)
trainer.train()

print("\nTraining complete.")
print(trainer.state.log_history)

In [ ]:
!pip install -U trl transformers accelerate datasets -q

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

model_name = "gpt2"
policy_model = AutoModelForCausalLM.from_pretrained(model_name)
ref_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

pref_data = Dataset.from_dict({
    "prompt": ["Explain gravity."],
    "chosen": ["Gravity is the force that attracts two masses toward each other."],
    "rejected": ["Gravity is when things fall because they feel like it."],
})

config = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    num_train_epochs=3,
    max_length=128,
    report_to=[],
)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=config,
    train_dataset=pref_data,
    processing_class=tokenizer,   # renamed from tokenizer= in current TRL
)
trainer.train()

print("\nTraining complete.")
print(trainer.state.log_history)